In [ ]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. File paths
# ------------------------------------------------------------

cathedral_path = "../cathedral_final_researched_clean_osm_area.csv"
castle_path = "../european_castles_osm_area_elevation_enriched.csv"
pyramid_path = "../pyramids_kaggle_clean_manual_duration_corrected.csv"

cathedrals = pd.read_csv(cathedral_path)
castles = pd.read_csv(castle_path)
pyramids = pd.read_csv(pyramid_path)


# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def num(series):
    """Convert a pandas Series to numeric, coercing bad values to NaN."""
    return pd.to_numeric(series, errors="coerce")


def optional(df, col, default=np.nan):
    """Safely return a column if present, otherwise a default-filled Series."""
    if col in df.columns:
        return df[col]
    return pd.Series([default] * len(df), index=df.index)


def log_positive(series):
    """Natural log for positive numeric values only."""
    series = num(series)
    return np.where(series > 0, np.log(series), np.nan)


def make_study_ids(df, prefix):
    """Create stable study IDs by monument class."""
    return [f"{prefix}_{i:04d}" for i in range(1, len(df) + 1)]


# ------------------------------------------------------------
# 3. Harmonize cathedral records
# ------------------------------------------------------------

cathedral_duration_confidence = pd.Series(pd.NA, index=cathedrals.index, dtype="object")
cathedral_duration_confidence[num(cathedrals["duration_years"]).notna()] = (
    "harmonized_recorded_span"
)

cathedral_out = pd.DataFrame({
    "study_id": make_study_ids(cathedrals, "CAT"),
    "source_row_id": optional(cathedrals, "study_id"),
    "monument_type": "cathedral",

    "name": cathedrals["cathedral_name"],
    "country_or_region": cathedrals["country"],
    "site_or_city": cathedrals["city"],

    "latitude": num(cathedrals["lat"]),
    "longitude": num(cathedrals["lon"]),

    "start_year": num(cathedrals["start_year"]),
    "end_year": num(cathedrals["end_year"]),
    "start_year_type": "recorded_ce",

    "duration_years_harmonized": num(cathedrals["duration_years"]),
    "duration_low_years": num(cathedrals["duration_years"]),
    "duration_high_years": num(cathedrals["duration_years"]),
    "duration_method": "openICPSR_duration_enriched",
    "duration_confidence": cathedral_duration_confidence,

    "elevation_m": num(cathedrals["elevation_m"]),

    "area_m2": num(cathedrals["area_m2_final"]),
    "area_source": optional(cathedrals, "area_m2_source"),
    "area_interpretation": "cathedral surface/footprint proxy",

    "height_m": num(optional(cathedrals, "height_m")),
    "volume_m3_or_proxy": num(optional(cathedrals, "volume_proxy")),

    "status": optional(cathedrals, "completed"),
    "notes": optional(cathedrals, "wikipedia_date_notes")
})

cathedral_out["log_area_m2"] = log_positive(cathedral_out["area_m2"])


# ------------------------------------------------------------
# 4. Harmonize castle records
# ------------------------------------------------------------

castle_duration_method = pd.Series(
    "wikidata_wikipedia_duration",
    index=castles.index,
    dtype="object"
)

if "manual_duration_fix_applied" in castles.columns:
    fixed = castles["manual_duration_fix_applied"].astype(str).str.lower().eq("true")
    castle_duration_method[fixed] = "manual_duration_fix_after_wikidata_wikipedia"

castle_out = pd.DataFrame({
    "study_id": make_study_ids(castles, "CAS"),
    "source_row_id": optional(castles, "wikidata_id"),
    "monument_type": "castle",

    "name": castles["name"],
    "country_or_region": castles["country"],
    "site_or_city": castles["admin_area"],

    "latitude": num(optional(castles, "latitude_final")).combine_first(
        num(castles["latitude"])
    ),
    "longitude": num(optional(castles, "longitude_final")).combine_first(
        num(castles["longitude"])
    ),

    "start_year": num(castles["start_year"]),
    "end_year": num(castles["end_year"]),
    "start_year_type": "recorded_or_wikipedia_enriched_ce",

    "duration_years_harmonized": num(castles["construction_duration_years"]),
    "duration_low_years": num(castles["construction_duration_years"]),
    "duration_high_years": num(castles["construction_duration_years"]),
    "duration_method": castle_duration_method,
    "duration_confidence": optional(castles, "manual_duration_fix_status"),

    "elevation_m": num(castles["elevation_m_final"]),

    "area_m2": num(castles["area_m2_final"]),
    "area_source": optional(castles, "area_source_final"),
    "area_interpretation": "castle mapped footprint/grounds/outline proxy",

    "height_m": num(optional(castles, "height_m_final")),
    "volume_m3_or_proxy": np.nan,

    "status": optional(castles, "castle_status_notes"),
    "notes": optional(castles, "duration_interpretation_note")
})

castle_out["log_area_m2"] = log_positive(castle_out["area_m2"])


# ------------------------------------------------------------
# 5. Harmonize pyramid records
# ------------------------------------------------------------
# Pyramids do not have uniform construction start years in the same way
# as cathedrals/castles. For chronological plots, we use approximate
# dynasty start years as BCE-negative proxy dates.
#
# These are not model features unless explicitly chosen later.
# They are chronological plotting proxies.

dynasty_start_proxy = {
    3: -2686,
    4: -2613,
    5: -2494,
    6: -2345,
    8: -2181,
    12: -1991,
    13: -1802,
    18: -1550,
}

dynasty_end_proxy = {
    3: -2613,
    4: -2494,
    5: -2345,
    6: -2181,
    8: -2160,
    12: -1802,
    13: -1650,
    18: -1292,
}

dynasty_numeric = num(pyramids["dynasty_final"]).astype("Int64")

# For classification, use direct/best duration when available.
# If not available, use construction_duration_high_years, which contains
# the reign-window upper-bound proxy for many pyramids.
pyramid_duration_harmonized = num(
    pyramids["construction_duration_best_years"]
).combine_first(
    num(pyramids["construction_duration_high_years"])
)

# Base area for pyramids is geometrically defined from base dimensions.
pyramid_base_area = num(pyramids["Base1 (m)"]) * num(pyramids["Base2 (m)"])

pyramid_out = pd.DataFrame({
    "study_id": make_study_ids(pyramids, "PYR"),
    "source_row_id": pyramids.index + 1,
    "monument_type": "pyramid",

    "name": pyramids["name_final"],
    "country_or_region": "Egypt",
    "site_or_city": pyramids["site_final"],

    "latitude": num(pyramids["Latitude"]),
    "longitude": num(pyramids["Longitude"]),

    "start_year": dynasty_numeric.map(dynasty_start_proxy).astype(float),
    "end_year": dynasty_numeric.map(dynasty_end_proxy).astype(float),
    "start_year_type": "dynasty_start_proxy_bce_negative",

    "duration_years_harmonized": pyramid_duration_harmonized,
    "duration_low_years": num(pyramids["construction_duration_low_years"]),
    "duration_high_years": num(pyramids["construction_duration_high_years"]),
    "duration_method": pyramids["duration_method"],
    "duration_confidence": pyramids["duration_confidence"],

    "elevation_m": num(pyramids["elevation_m"]),

    "area_m2": pyramid_base_area,
    "area_source": "kaggle_base1_x_base2",
    "area_interpretation": "pyramid base area from base dimensions",

    "height_m": num(pyramids["height_m_final"]),
    "volume_m3_or_proxy": num(pyramids["volume_m3_final"]),

    "status": pyramids["status_final"],
    "notes": pyramids["duration_caution"]
})

pyramid_out["log_area_m2"] = log_positive(pyramid_out["area_m2"])


# ------------------------------------------------------------
# 6. Combine into one harmonized dataset
# ------------------------------------------------------------

final_columns = [
    "study_id",
    "source_row_id",
    "monument_type",
    "name",
    "country_or_region",
    "site_or_city",
    "latitude",
    "longitude",
    "start_year",
    "end_year",
    "start_year_type",
    "duration_years_harmonized",
    "duration_low_years",
    "duration_high_years",
    "duration_method",
    "duration_confidence",
    "elevation_m",
    "area_m2",
    "log_area_m2",
    "area_source",
    "area_interpretation",
    "height_m",
    "volume_m3_or_proxy",
    "status",
    "notes"
]

combined_all = pd.concat(
    [
        cathedral_out[final_columns],
        castle_out[final_columns],
        pyramid_out[final_columns]
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# 7. Complete-case dataset for separability analysis
# ------------------------------------------------------------

required_shared_features = [
    "duration_years_harmonized",
    "elevation_m",
    "area_m2",
    "log_area_m2"
]

# First, keep only rows that have all shared modeling features.
combined_complete_case = combined_all.dropna(
    subset=required_shared_features
).copy()

# Remove non-positive duration records.
# These usually reflect single-year or unresolved construction records,
# not meaningful evidence of instantaneous construction.
combined_complete_case = combined_complete_case[
    combined_complete_case["duration_years_harmonized"] > 0
].copy()

# Remove post-1750 cathedral and castle records.
# These later records are more likely to reflect restoration, revival,
# modern completion, or heritage-phase construction histories rather than
# the pre-modern monument-building logic central to the comparison.
combined_complete_case = combined_complete_case[
    ~(
        combined_complete_case["monument_type"].isin(["cathedral", "castle"]) &
        (combined_complete_case["start_year"] > 1750)
    )
].copy()

# Optional but useful: reset the index after filtering.
combined_complete_case = combined_complete_case.reset_index(drop=True)

# ------------------------------------------------------------
# 8. Sanity checks
# ------------------------------------------------------------

print("Full harmonized dataset:")
print(combined_all["monument_type"].value_counts())
print("Total:", len(combined_all))

print("\nComplete-case dataset:")
print(combined_complete_case["monument_type"].value_counts())
print("Total:", len(combined_complete_case))

print("\nMissing required features in complete-case dataset:")
print(combined_complete_case[required_shared_features].isna().sum())


# ------------------------------------------------------------
# 9. Save outputs
# ------------------------------------------------------------

combined_all.to_csv(
    "monuments_harmonized_all.csv",
    index=False
)

combined_complete_case.to_csv(
    "monuments_harmonized_complete_case_positive_duration_pre1750.csv",
    index=False
)

Full harmonized dataset:
monument_type
cathedral    313
castle       112
pyramid       62
Name: count, dtype: int64
Total: 487

Complete-case dataset:
monument_type
cathedral    297
castle        76
pyramid       43
Name: count, dtype: int64
Total: 416

Missing required features in complete-case dataset:
duration_years_harmonized    0
elevation_m                  0
area_m2                      0
log_area_m2                  0
dtype: int64


In [3]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from io import StringIO
from difflib import SequenceMatcher


# ============================================================
# 1. File paths
# ============================================================

DATA_PATH = Path("monuments_harmonized_complete_case_positive_duration_pre1750.csv")

CATHEDRAL_NOTES_PATH = Path("cathedral_manual_correction_notes_combined.md")
CASTLE_NOTES_PATH = Path("castle_manual_correction_notes.md")

OUT_CORRECTED = Path("monuments_harmonized_cathedral_castle_manual_duration_corrected.csv")
OUT_PARSED = Path("parsed_cathedral_castle_manual_corrections.csv")
OUT_APPLIED_AUDIT = Path("applied_cathedral_castle_manual_correction_audit.csv")
OUT_DROP_AUDIT = Path("dropped_cathedral_castle_manual_correction_audit.csv")
OUT_UNMATCHED = Path("unmatched_cathedral_castle_manual_corrections.csv")
OUT_KEPT_NO_NUMERIC = Path("kept_rows_without_numeric_correction.csv")


# ============================================================
# 2. Settings
# ============================================================

FUZZY_MATCH_THRESHOLD = 0.35

# Egeskov is corrected to 1554–1554, duration 0.
# For modeling, positive-duration files should usually drop zero-duration rows.
DROP_ZERO_OR_NEGATIVE_DURATION = True

# Preserve the same pre-1750 logic by corrected start year.
DROP_CATHEDRAL_CASTLE_START_AFTER_1750 = True


# ============================================================
# 3. Load data and notes
# ============================================================

df = pd.read_csv(DATA_PATH)

cathedral_notes = CATHEDRAL_NOTES_PATH.read_text(encoding="utf-8", errors="ignore")
castle_notes = CASTLE_NOTES_PATH.read_text(encoding="utf-8", errors="ignore")

all_notes = {
    "cathedral": cathedral_notes,
    "castle": castle_notes,
}

print("Loaded dataset:", df.shape)
print(df["monument_type"].value_counts())


# ============================================================
# 4. Preserve original fields
# ============================================================

required_cols = ["monument_type", "name", "site_or_city", "start_year", "end_year", "duration_years_harmonized"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in dataset: {missing}")

df["start_year_original"] = pd.to_numeric(df["start_year"], errors="coerce")
df["end_year_original"] = pd.to_numeric(df["end_year"], errors="coerce")
df["duration_years_original"] = pd.to_numeric(df["duration_years_harmonized"], errors="coerce")

df["start_year_corrected"] = df["start_year_original"]
df["end_year_corrected"] = df["end_year_original"]
df["duration_years_corrected"] = df["duration_years_original"]

df["date_correction_status"] = "unchanged"
df["date_basis_manual"] = ""
df["date_confidence_manual"] = ""
df["manual_correction_notes"] = ""

df["drop_from_manual_corrected_file"] = False
df["manual_drop_reason"] = ""


# ============================================================
# 5. Helper functions
# ============================================================

def normalize_text(x):
    x = str(x).lower()
    x = x.replace("œ", "oe").replace("æ", "ae")
    x = x.replace("’", "'").replace("`", "'")
    x = re.sub(r"\([^)]*\)", " ", x)
    x = re.sub(r"[^a-z0-9à-ÿ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def similarity(a, b):
    return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()


def split_md_row(line):
    line = line.strip()
    if not (line.startswith("|") and line.endswith("|")):
        return None

    cells = [c.strip() for c in line.strip("|").split("|")]

    # Skip markdown separator rows
    if all(re.fullmatch(r":?-{2,}:?", c.replace(" ", "")) for c in cells):
        return None

    return cells


def parse_raw_range(raw):
    raw = str(raw).replace("–", "-").replace("—", "-")
    m = re.search(r"(-?\d{1,4})\s*-\s*(-?\d{1,4})", raw)
    if not m:
        return np.nan, np.nan
    return float(m.group(1)), float(m.group(2))


def is_keep_status(status):
    s = normalize_text(status)
    return "keep" in s


def is_drop_status(status):
    s = normalize_text(status)

    if "keep" in s:
        return False

    return any(tok in s for tok in [
        "review",
        "drop",
        "exclude",
        "not ready",
        "drop review",
        "review drop",
        "drop for premodern model",
    ])


def extract_numeric_ranges(text):
    """
    Extract ranges like:
    1194–1260
    c. 1150–1465
    1151/1153–1191
    1277/1278–1511
    1248–1880 if full; 1248–1473 if medieval-only
    """
    text = str(text)
    text = text.replace("**", "")
    text = text.replace("–", "-").replace("—", "-")
    text = text.replace("c.", "").replace("ca.", "").replace("approx.", "")

    ranges = []

    pattern = r"(-?\d{3,4})(?:/\d{2,4})?\s*-\s*(-?\d{3,4})(?:/\d{2,4})?"
    for m in re.finditer(pattern, text):
        start = int(m.group(1))
        end = int(m.group(2))

        if end >= start:
            ranges.append((float(start), float(end)))

    return ranges


def choose_best_range(corrected_text, ranges):
    """
    Prefer explicit modeling / premodern recommendation.
    Otherwise prefer ranges ending <= 1750.
    Otherwise use first valid range.
    """
    if not ranges:
        return np.nan, np.nan

    text = str(corrected_text).replace("–", "-").replace("—", "-").replace("**", "")
    lower = text.lower()

    m = re.search(
        r"(?:for your premodern model|for modeling|modeling core|use)\D{0,100}?(\d{3,4})\s*-\s*(\d{3,4})",
        lower
    )
    if m:
        start = float(m.group(1))
        end = float(m.group(2))
        if end >= start:
            return start, end

    premodern = [(s, e) for s, e in ranges if e <= 1750]
    if premodern:
        return premodern[-1]

    return ranges[0]


def infer_confidence(text, status, correction_status=None):
    joined = normalize_text(f"{text} {status} {correction_status}")

    if "zero" in joined or "single year" in joined:
        return "zero_duration_flag"

    if any(w in joined for w in [
        "broad",
        "proxy",
        "phase choice",
        "choose phase",
        "define phase",
        "provisional",
        "approx",
        "keep review",
    ]):
        return "medium_broad_or_phase_choice"

    return "high_numeric"


def find_best_df_match(data, monument_type, monument, raw_start, raw_end, site_or_city=None):
    candidates = data[data["monument_type"].eq(monument_type)].copy()

    if pd.notna(raw_start) and pd.notna(raw_end):
        candidates = candidates[
            np.isclose(pd.to_numeric(candidates["start_year_original"], errors="coerce"), raw_start)
            & np.isclose(pd.to_numeric(candidates["end_year_original"], errors="coerce"), raw_end)
        ].copy()

    if candidates.empty:
        return None, None

    target = normalize_text(f"{monument} {site_or_city or ''}")

    scored = []
    for idx, row in candidates.iterrows():
        haystack = " ".join([
            str(row.get("name", "")),
            str(row.get("site_or_city", "")),
            str(row.get("country_or_region", "")),
        ])
        scored.append((idx, similarity(target, haystack)))

    best_idx, best_score = max(scored, key=lambda x: x[1])
    return best_idx, best_score


# ============================================================
# 6. Parse markdown correction tables
#    Mainly used for cathedral notes.
# ============================================================

def parse_markdown_tables(notes_text, default_monument_type):
    parsed_rows = []
    mode = None
    headers = None

    for line in notes_text.splitlines():
        low = line.lower()

        if "## keep + correct" in low:
            mode = "keep"
            headers = None
            continue

        if "## exclude / review later" in low or "## drop / review later" in low:
            mode = "drop"
            headers = None
            continue

        cells = split_md_row(line)
        if cells is None:
            continue

        norm_headers_candidate = [normalize_text(c) for c in cells]

        if any(("monument" in c or "castle" == c) for c in norm_headers_candidate) and any(
            ("raw" in c or "current" in c or "corrected start" in c) for c in norm_headers_candidate
        ):
            headers = norm_headers_candidate
            continue

        if headers is None:
            continue

        monument_idx = raw_idx = corrected_idx = keep_idx = notes_idx = None

        for i, h in enumerate(headers):
            if "monument" in h or h == "castle":
                monument_idx = i
            elif "raw" in h or "current" in h:
                raw_idx = i
            elif "corrected" in h or "suggested" in h:
                corrected_idx = i
            elif "keep" in h or "status" in h:
                keep_idx = i
            elif "note" in h or "basis" in h or "reason" in h:
                notes_idx = i

        if monument_idx is None or monument_idx >= len(cells):
            continue

        monument = cells[monument_idx]
        status = cells[keep_idx] if keep_idx is not None and keep_idx < len(cells) else ""
        note_text = cells[notes_idx] if notes_idx is not None and notes_idx < len(cells) else ""
        corrected_text = cells[corrected_idx] if corrected_idx is not None and corrected_idx < len(cells) else ""

        # Standard cathedral tables have raw/current dates.
        raw_start = raw_end = np.nan
        if raw_idx is not None and raw_idx < len(cells):
            raw_start, raw_end = parse_raw_range(cells[raw_idx])

        # Skip tables that do not have raw dates; castle CSV block handles those.
        if pd.isna(raw_start) or pd.isna(raw_end):
            continue

        if mode == "drop":
            action = "drop"
        elif keep_idx is not None:
            if is_keep_status(status):
                action = "keep"
            elif is_drop_status(status):
                action = "drop"
            else:
                action = "review"
        elif mode == "keep":
            action = "keep"
        else:
            action = "review"

        ranges = extract_numeric_ranges(corrected_text)
        corrected_start, corrected_end = choose_best_range(corrected_text, ranges)

        parsed_rows.append({
            "monument_type": default_monument_type,
            "monument_from_notes": monument,
            "site_or_city_from_notes": "",
            "raw_start": raw_start,
            "raw_end": raw_end,
            "corrected_start": corrected_start,
            "corrected_end": corrected_end,
            "corrected_duration": corrected_end - corrected_start if pd.notna(corrected_start) and pd.notna(corrected_end) else np.nan,
            "action": action,
            "status_from_notes": status,
            "correction_status_from_notes": "",
            "corrected_text": corrected_text,
            "notes_from_notes": note_text,
            "source_parser": "markdown_table",
            "date_confidence_manual": infer_confidence(corrected_text, status),
        })

    return parsed_rows


# ============================================================
# 7. Parse CSV correction blocks
#    Mainly used for castle notes.
# ============================================================

def parse_csv_code_blocks(notes_text, default_monument_type=None):
    parsed_rows = []

    blocks = re.findall(r"```csv\s*(.*?)```", notes_text, flags=re.DOTALL | re.IGNORECASE)

    for block in blocks:
        block = block.strip()
        if not block.startswith("monument_type,"):
            continue

        temp = pd.read_csv(StringIO(block))

        for _, r in temp.iterrows():
            monument_type = str(r.get("monument_type", default_monument_type)).strip()
            name = str(r.get("name", "")).strip()
            site = str(r.get("site_or_city", "")).strip()

            raw_start = pd.to_numeric(r.get("raw_start_year", np.nan), errors="coerce")
            raw_end = pd.to_numeric(r.get("raw_end_year", np.nan), errors="coerce")
            corrected_start = pd.to_numeric(r.get("corrected_start_year", np.nan), errors="coerce")
            corrected_end = pd.to_numeric(r.get("corrected_end_year", np.nan), errors="coerce")
            corrected_duration = pd.to_numeric(r.get("corrected_duration_years", np.nan), errors="coerce")

            correction_status = str(r.get("correction_status", "")).strip()
            correction_note = str(r.get("correction_note", "")).strip()

            if "drop" in normalize_text(correction_status):
                action = "drop"
            else:
                action = "keep"

            parsed_rows.append({
                "monument_type": monument_type,
                "monument_from_notes": name,
                "site_or_city_from_notes": site,
                "raw_start": raw_start,
                "raw_end": raw_end,
                "corrected_start": corrected_start,
                "corrected_end": corrected_end,
                "corrected_duration": corrected_duration,
                "action": action,
                "status_from_notes": correction_status,
                "correction_status_from_notes": correction_status,
                "corrected_text": f"{corrected_start}–{corrected_end}" if pd.notna(corrected_start) and pd.notna(corrected_end) else "",
                "notes_from_notes": correction_note,
                "source_parser": "csv_code_block",
                "date_confidence_manual": infer_confidence(correction_note, correction_status, correction_status),
            })

    return parsed_rows


# ============================================================
# 8. Parse both note files
# ============================================================

parsed_rows = []

# Cathedral notes: markdown tables
parsed_rows.extend(parse_markdown_tables(cathedral_notes, "cathedral"))

# Castle notes: CSV code block is the authoritative correction table
parsed_rows.extend(parse_csv_code_blocks(castle_notes, "castle"))

parsed = pd.DataFrame(parsed_rows)

if parsed.empty:
    raise ValueError("No corrections parsed from notes files.")

parsed = parsed.drop_duplicates(
    subset=[
        "monument_type",
        "monument_from_notes",
        "site_or_city_from_notes",
        "raw_start",
        "raw_end",
        "corrected_start",
        "corrected_end",
        "action",
    ]
)

parsed.to_csv(OUT_PARSED, index=False)

print("\nParsed corrections:")
print(parsed.groupby(["monument_type", "action"]).size())


# ============================================================
# 9. Apply drops first
# ============================================================

drop_audit = []
applied_audit = []
unmatched = []
kept_no_numeric = []

for _, row in parsed[parsed["action"].eq("drop")].iterrows():
    idx, score = find_best_df_match(
        df,
        row["monument_type"],
        row["monument_from_notes"],
        row["raw_start"],
        row["raw_end"],
        row.get("site_or_city_from_notes", ""),
    )

    if idx is None or score < FUZZY_MATCH_THRESHOLD:
        unmatched.append({**row.to_dict(), "unmatched_reason": "drop_no_match", "similarity": score})
        continue

    df.loc[idx, "drop_from_manual_corrected_file"] = True
    df.loc[idx, "manual_drop_reason"] = "manual_notes_drop_or_review"

    drop_audit.append({
        "row_index": idx,
        "study_id": df.loc[idx, "study_id"] if "study_id" in df.columns else "",
        "monument_type": df.loc[idx, "monument_type"],
        "name": df.loc[idx, "name"],
        "site_or_city": df.loc[idx, "site_or_city"],
        "raw_start": df.loc[idx, "start_year_original"],
        "raw_end": df.loc[idx, "end_year_original"],
        "matched_to": row["monument_from_notes"],
        "similarity": score,
        "reason": row["notes_from_notes"],
    })


# ============================================================
# 10. Apply keep corrections
# ============================================================

for _, row in parsed[parsed["action"].eq("keep")].iterrows():
    idx, score = find_best_df_match(
        df[~df["drop_from_manual_corrected_file"]],
        row["monument_type"],
        row["monument_from_notes"],
        row["raw_start"],
        row["raw_end"],
        row.get("site_or_city_from_notes", ""),
    )

    if idx is None or score < FUZZY_MATCH_THRESHOLD:
        unmatched.append({**row.to_dict(), "unmatched_reason": "keep_no_match", "similarity": score})
        continue

    # Keep rows with broad/no numeric notes but do not fake numeric correction.
    if pd.isna(row["corrected_start"]) or pd.isna(row["corrected_end"]):
        df.loc[idx, "date_correction_status"] = "kept_no_numeric_correction"
        df.loc[idx, "date_confidence_manual"] = "broad_or_no_numeric"
        df.loc[idx, "manual_correction_notes"] = str(row["corrected_text"]) + " | " + str(row["notes_from_notes"])

        kept_no_numeric.append({
            "row_index": idx,
            "study_id": df.loc[idx, "study_id"] if "study_id" in df.columns else "",
            "monument_type": df.loc[idx, "monument_type"],
            "name": df.loc[idx, "name"],
            "site_or_city": df.loc[idx, "site_or_city"],
            "raw_start": df.loc[idx, "start_year_original"],
            "raw_end": df.loc[idx, "end_year_original"],
            "matched_to": row["monument_from_notes"],
            "corrected_text": row["corrected_text"],
            "notes": row["notes_from_notes"],
            "similarity": score,
        })
        continue

    new_start = float(row["corrected_start"])
    new_end = float(row["corrected_end"])
    new_duration = float(row["corrected_duration"]) if pd.notna(row["corrected_duration"]) else new_end - new_start

    if new_end < new_start:
        unmatched.append({**row.to_dict(), "unmatched_reason": "invalid_corrected_range", "similarity": score})
        continue

    old_start = df.loc[idx, "start_year_corrected"]
    old_end = df.loc[idx, "end_year_corrected"]
    old_duration = df.loc[idx, "duration_years_corrected"]

    df.loc[idx, "start_year_corrected"] = new_start
    df.loc[idx, "end_year_corrected"] = new_end
    df.loc[idx, "duration_years_corrected"] = new_duration

    if new_duration == 0:
        df.loc[idx, "date_correction_status"] = "corrected_zero_duration_flag"
    else:
        df.loc[idx, "date_correction_status"] = "corrected"

    df.loc[idx, "date_basis_manual"] = "manual_notes_main_present_or_substantially_preserved_building"
    df.loc[idx, "date_confidence_manual"] = row["date_confidence_manual"]
    df.loc[idx, "manual_correction_notes"] = str(row["corrected_text"]) + " | " + str(row["notes_from_notes"])

    applied_audit.append({
        "row_index": idx,
        "study_id": df.loc[idx, "study_id"] if "study_id" in df.columns else "",
        "monument_type": df.loc[idx, "monument_type"],
        "name": df.loc[idx, "name"],
        "site_or_city": df.loc[idx, "site_or_city"],
        "matched_to": row["monument_from_notes"],
        "similarity": score,
        "old_start": old_start,
        "old_end": old_end,
        "old_duration": old_duration,
        "new_start": new_start,
        "new_end": new_end,
        "new_duration": new_duration,
        "confidence": row["date_confidence_manual"],
        "notes": row["notes_from_notes"],
    })


applied_audit_df = pd.DataFrame(applied_audit)
drop_audit_df = pd.DataFrame(drop_audit)
unmatched_df = pd.DataFrame(unmatched)
kept_no_numeric_df = pd.DataFrame(kept_no_numeric)


# ============================================================
# 11. Build corrected output
# ============================================================

df_corrected = df[~df["drop_from_manual_corrected_file"]].copy()

df_corrected["start_year"] = df_corrected["start_year_corrected"]
df_corrected["end_year"] = df_corrected["end_year_corrected"]
df_corrected["duration_years_harmonized"] = df_corrected["duration_years_corrected"]

if "duration_low_years" in df_corrected.columns:
    df_corrected["duration_low_years"] = df_corrected["duration_years_corrected"]

if "duration_high_years" in df_corrected.columns:
    df_corrected["duration_high_years"] = df_corrected["duration_years_corrected"]

corrected_mask = df_corrected["date_correction_status"].isin(["corrected", "corrected_zero_duration_flag"])

if "duration_method" in df_corrected.columns:
    df_corrected.loc[corrected_mask, "duration_method"] = "manual_researched_start_end_correction"

if "duration_confidence" in df_corrected.columns:
    df_corrected.loc[corrected_mask, "duration_confidence"] = df_corrected.loc[corrected_mask, "date_confidence_manual"]


# Optional modeling filters
if DROP_ZERO_OR_NEGATIVE_DURATION:
    df_corrected = df_corrected[
        pd.to_numeric(df_corrected["duration_years_harmonized"], errors="coerce") > 0
    ].copy()

if DROP_CATHEDRAL_CASTLE_START_AFTER_1750:
    df_corrected = df_corrected[
        ~(
            df_corrected["monument_type"].isin(["cathedral", "castle"])
            & (pd.to_numeric(df_corrected["start_year"], errors="coerce") > 1750)
        )
    ].copy()

df_corrected = df_corrected.reset_index(drop=True)


# ============================================================
# 12. Save outputs
# ============================================================

df_corrected.to_csv(OUT_CORRECTED, index=False)
applied_audit_df.to_csv(OUT_APPLIED_AUDIT, index=False)
drop_audit_df.to_csv(OUT_DROP_AUDIT, index=False)
unmatched_df.to_csv(OUT_UNMATCHED, index=False)
kept_no_numeric_df.to_csv(OUT_KEPT_NO_NUMERIC, index=False)


# ============================================================
# 13. Summary
# ============================================================

print("\nSaved corrected file:")
print(OUT_CORRECTED)

print("\nCorrected dataset shape:", df_corrected.shape)

print("\nClass counts after correction/drop:")
print(df_corrected["monument_type"].value_counts())

print("\nApplied numeric corrections:", len(applied_audit_df))
if len(applied_audit_df):
    print(applied_audit_df["monument_type"].value_counts())

print("\nDropped review/drop rows:", len(drop_audit_df))
if len(drop_audit_df):
    print(drop_audit_df["monument_type"].value_counts())

print("\nUnmatched correction rows:", len(unmatched_df))
if len(unmatched_df):
    print(unmatched_df[["monument_type", "monument_from_notes", "raw_start", "raw_end", "action", "unmatched_reason", "similarity"]].head(30))

print("\nKept rows without numeric correction:", len(kept_no_numeric_df))
if len(kept_no_numeric_df):
    print(kept_no_numeric_df[["monument_type", "name", "site_or_city", "matched_to", "corrected_text"]].head(30))

print("\nCorrection status counts:")
print(df_corrected["date_correction_status"].value_counts(dropna=False))

print("\nDuration consistency check:")
tmp = df_corrected.copy()
tmp["duration_from_dates_check"] = (
    pd.to_numeric(tmp["end_year"], errors="coerce")
    - pd.to_numeric(tmp["start_year"], errors="coerce")
)
tmp["duration_difference_check"] = (
    pd.to_numeric(tmp["duration_years_harmonized"], errors="coerce")
    - tmp["duration_from_dates_check"]
)

print(
    tmp.groupby("monument_type")["duration_difference_check"]
    .agg(["count", "min", "max", "mean"])
)

print("\nAudit files written:")
print(OUT_PARSED)
print(OUT_APPLIED_AUDIT)
print(OUT_DROP_AUDIT)
print(OUT_UNMATCHED)
print(OUT_KEPT_NO_NUMERIC)

Loaded dataset: (416, 25)
monument_type
cathedral    297
castle        76
pyramid       43
Name: count, dtype: int64

Parsed corrections:
monument_type  action
castle         drop        1
               keep       46
cathedral      drop       59
               keep      243
dtype: int64

Saved corrected file:
monuments_harmonized_cathedral_castle_manual_duration_corrected.csv

Corrected dataset shape: (384, 37)

Class counts after correction/drop:
monument_type
cathedral    268
castle        73
pyramid       43
Name: count, dtype: int64

Applied numeric corrections: 139
monument_type
cathedral    93
castle       46
Name: count, dtype: int64

Dropped review/drop rows: 30
monument_type
cathedral    29
castle        1
Name: count, dtype: int64

Unmatched correction rows: 146
   monument_type                              monument_from_notes  raw_start  \
0      cathedral                                Trapani Cathedral     1421.0   
1      cathedral                                Châlons 

In [4]:
from pathlib import Path
import pandas as pd

# Load the corrected modeling CSV
input_path = Path("monuments_harmonized_cathedral_castle_manual_duration_corrected.csv")

# Save a new version with Pevensey removed
output_path = Path("monuments_harmonized_cathedral_castle_manual_duration_corrected.csv")

df = pd.read_csv(input_path)

# Remove Pevensey Castle:
# Its start date reflects the Roman shore-fort phase, while its later endpoint reflects
# medieval castle reuse/development. This makes it a Roman-fort-origin outlier rather
# than a clean medieval European castle construction-duration record.
remove_mask = (
    (df["monument_type"].astype(str).str.lower() == "castle")
    & (df["name"].astype(str).str.lower() == "pevensey castle")
)

df_no_pevensey = df.loc[~remove_mask].copy()

df_no_pevensey.to_csv(output_path, index=False)

print(f"Original rows: {len(df)}")
print(f"Rows removed: {remove_mask.sum()}")
print(f"Remaining rows: {len(df_no_pevensey)}")
print(f"Saved to: {output_path}")

Original rows: 384
Rows removed: 1
Remaining rows: 383
Saved to: monuments_harmonized_cathedral_castle_manual_duration_corrected.csv
